# Insurance Premium Billing & Collections — SQL Analysis

This notebook uses SQL (via SQLite) to analyze the reconciliation dataset
generated in `data_generation.ipynb`. The goal is to answer the same
questions a Surety Premium Collections Analyst would ask day-to-day:
which invoices are past due, which agencies carry the most outstanding
balance, and how payment behavior varies by scenario.

##  Load Data into SQLite

Load the reconciliation CSV into an in-memory SQLite database so we can
query it with SQL instead of pandas.

In [18]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(":memory:")
recon = pd.read_csv("../data/raw/reconciliation.csv")
recon.to_sql("reconciliation", conn, index=False, if_exists="replace")

recon.head()

,invoice_id,policy_number,agency_id,invoice_date,premium_billed,due_date,profile,scenario,payment_amount,agency_name,line_of_business,balance,status,days_past_due,aging_bucket
0,INV-0001,POL-1000,A01,2026-06-01,1399.58,2026-07-01,reliable,late,1399.58,Summit Insurance Group,Commercial Property,0.00,Paid,76,Paid
1,INV-0002,POL-1000,A01,2026-07-01,1399.58,2026-07-31,reliable,on_time,1399.58,Summit Insurance Group,Commercial Property,0.00,Paid,46,Paid
2,INV-0003,POL-1000,A01,2026-08-01,1399.58,2026-08-31,reliable,on_time,1399.58,Summit Insurance Group,Commercial Property,0.00,Paid,15,Paid
3,INV-0004,POL-1001,A08,2026-06-01,533.17,2026-07-01,slow,short,392.50,Ironwood Insurance Agency,Commercial Auto,140.67,Past Due,76,60+ Days
4,INV-0005,POL-1001,A08,2026-07-01,533.17,2026-07-31,slow,late,533.17,Ironwood Insurance Agency,Commercial Auto,0.00,Paid,46,Paid


##  Aging Summary

How much outstanding balance falls into each aging bucket
(Current, 1-30 Days, 31-60 Days, 60+ Days)? This is the core report
a collections team reviews to prioritize follow-up.

In [5]:
query1 = """
SELECT aging_bucket,
       COUNT(*) as invoice_count,
       ROUND(SUM(balance), 2) as total_balance
FROM reconciliation
GROUP BY aging_bucket
ORDER BY total_balance DESC
"""
pd.read_sql(query1, conn)

,aging_bucket,invoice_count,total_balance
0,1-30 Days,21,40205.45
1,31-60 Days,11,15857.91
2,60+ Days,10,6448.03
3,Paid,108,0.00


##  Top Past-Due Agencies

Which agencies carry the highest outstanding balance right now?
This identifies where collections effort should be focused first.

In [6]:
query2 = """
SELECT agency_name,
       COUNT(*) as invoice_count,
       ROUND(SUM(balance), 2) as total_past_due
FROM reconciliation
WHERE status = 'Past Due'
GROUP BY agency_name
ORDER BY total_past_due DESC
LIMIT 5
"""
pd.read_sql(query2, conn)

,agency_name,invoice_count,total_past_due
0,Ironwood Insurance Agency,13,25845.73
1,Sterling Benefits & Risk,12,13699.48
2,Northgate Underwriters,6,10539.74
3,Cardinal Insurance Group,4,5265.83
4,Bayview Risk Advisors,4,2792.85


##  Payment Behavior by Scenario

Compare average outstanding balance across the three payment scenarios
(on_time, late, short) to confirm the simulated payment behavior is
reflected accurately in the reconciliation results.

In [7]:
query3 = """
SELECT scenario,
       COUNT(*) as invoice_count,
       ROUND(AVG(balance), 2) as avg_balance
FROM reconciliation
GROUP BY scenario
ORDER BY avg_balance DESC
"""
pd.read_sql(query3, conn)

,scenario,invoice_count,avg_balance
0,late,47,960.13
1,short,23,755.88
2,on_time,80,0.00


##  Status Breakdown

A quick count of invoices by status (Paid, Current, Past Due) —
confirms totals line up with the aging summary above.

In [8]:
query4 = """
SELECT status,
       COUNT(*) as invoice_count,
       ROUND(SUM(balance), 2) as total_balance
FROM reconciliation
GROUP BY status
ORDER BY invoice_count DESC
"""
pd.read_sql(query4, conn)

,status,invoice_count,total_balance
0,Paid,108,0.00
1,Past Due,42,62511.39


##  Line of Business Breakdown

Does outstanding balance skew toward a particular line of business
(Commercial Auto, General Liability, Commercial Property)? Useful for
spotting whether risk concentrates in a specific policy type.

In [13]:
print(recon.columns.tolist())

['invoice_id', 'policy_number', 'agency_id', 'invoice_date', 'premium_billed', 'due_date', 'profile', 'scenario', 'payment_amount', 'agency_name', 'balance', 'status', 'days_past_due', 'aging_bucket']


In [19]:
query5 = """
SELECT line_of_business,
       COUNT(*) as invoice_count,
       ROUND(SUM(balance), 2) as total_balance
FROM reconciliation
WHERE status = 'Past Due'
GROUP BY line_of_business
ORDER BY total_balance DESC
"""
pd.read_sql(query5, conn)

,line_of_business,invoice_count,total_balance
0,Commercial Auto,19,27849.92
1,Commercial Property,16,24848.54
2,General Liability,7,9812.93
